In [1]:
# from sklearn.feature_extraction.text import CountVectorizer
# import joblib
# loaded_model = joblib.load('classifier_model.pkl')
# vectorizer = CountVectorizer()
# words=['politics','business','tech','sport','entertaiment']
# # Функция для получения текстового ввода с клавиатуры и классификации
# def classify_input_text():
#     new_text = input('Введите текст для классификации: ')
#     new_text_vectorized = vectorizer.transform([new_text])
#     predicted_label = loaded_model.predict(new_text_vectorized)
#     print('Предсказанная метка для введенного текста:', words[predicted_label[0]])

# classify_input_text()

In [2]:
!pip install -U datasets transformers

In [3]:
import pandas as pd
import numpy as np
from collections import defaultdict
data = pd.read_csv('bbc-news-data.csv', sep='\t')

In [4]:
#data.labels.value_counts()

In [5]:
titles =set(data['category'])
word_cat = defaultdict(int)
c = 0
for p in titles:
  word_cat[p] = c
  c += 1
#word_cat
data['category'] = data.apply(lambda x: word_cat[x['category']], axis = 1)
X = data['content'].values
print(X)
print(titles)
data =data.rename(columns={'category':'labels'})

[' Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.  The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.  Time Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, the company said AOL\'s underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues. It hopes to increase subscribers by offering the online service free to TimeWarner internet customers and will try to sign up AOL\'s existing customers for high

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling


In [7]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from datasets import Dataset
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertForSequenceClassification, AdamW

# Загружаем предварительно обученную модель BERT и токенизатор
model_name = 'bert-base-uncased'
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=5)
tokenizer = BertTokenizer.from_pretrained(model_name)



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
def tokenize_function(examples):
    return tokenizer(examples['content'], padding=True, truncation=True)

dataset = Dataset.from_pandas(data)
tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenized_dataset.save_to_disk('dataset.hf')

Map:   0%|          | 0/2225 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2225 [00:00<?, ? examples/s]

In [27]:
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.2)

In [29]:
tokenized_dataset['test']

Dataset({
    features: ['labels', 'filename', 'title', 'content', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 445
})

In [11]:
from datasets import load_from_disk
X_train = load_from_disk('dataset.hf')

In [24]:
type(X_train)

dict

In [12]:
#%pip install transformers[torch]
%pip install accelerate -U

In [13]:
import accelerate

accelerate.__version__

from datasets import load_metric
metric = load_metric('accuracy')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

<ipython-input-13-5c4084152759>:6: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric = load_metric('accuracy')


In [34]:
 #Training arguments
from transformers import DataCollatorWithPadding

training_args = TrainingArguments(
    # add metrics , accuracy
    output_dir="./output",
    overwrite_output_dir=True,
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    save_steps=300,
    # eval_steps=100,
    save_total_limit=2,
    do_eval=True,
    evaluation_strategy='epoch'

)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    train_dataset=tokenized_dataset['train'],
    compute_metrics=compute_metrics,
    eval_dataset = tokenized_dataset['test']
)

# Train the model
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.126830,0.982022
2,No log,0.099744,0.986517
3,No log,0.139100,0.979775
4,No log,0.110303,0.986517
5,0.066600,0.115118,0.986517
6,0.066600,0.116506,0.986517


TrainOutput(global_step=672, training_loss=0.049641197647101114, metrics={'train_runtime': 1104.6633, 'train_samples_per_second': 9.668, 'train_steps_per_second': 0.608, 'total_flos': 2810101761515520.0, 'train_loss': 0.049641197647101114, 'epoch': 6.0})

In [36]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [35]:
model.save_pretrained('/content/model')

In [16]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.metrics import accuracy_score

In [21]:
# from transformers import BertTokenizer, BertForSequenceClassification
# import torch
# Инициализация предварительно обученной модели BERT для классификации

input_text = input("Введите текст для классификации: ")
names = list(titles)
# Предобработка введенного текста
# inputs = tokenizer(X_test, return_tensors='pt', padding=True, truncation=True)

# Режим оценки (evaluation mode)
model.eval()

# Применение модели к входным данным
with torch.no_grad():
    outputs = model(**[X_test])

# Получение предсказанного класса
predicted_class_index = torch.argmax(outputs.logits).item()
print("Предсказанный класс:", names[predicted_class_index])


Введите текст для классификации: sport


TypeError: ignored